# biblia-audio-baixar.ipynb — Bíblia em áudio (WEB / David Williams) pro Drive

Baixa a gravação **World English Bible** feita por David Williams (domínio
público, AudioTreasure), descompacta e salva no Drive **um mp3 por capítulo**,
já renomeado pro padrão do projeto (`40_Matt_02.mp3`).

Roda **uma vez**. Depois disso qualquer compilação acha o capítulo que precisar
sem você subir nada — o passo "fornecer o áudio" some do fluxo.

**Por que renomear:** os nomes na fonte são irregulares (`40_Matthew01` sem
underscore, `19_Psalm_001` com três dígitos, `25_Lamentations03`,
`22_Song_of_Soloman_01`). A tabela canônica em `modulos/biblia_livros.py`
resolve cada caso e devolve o nome do projeto.

**Tamanho:** ~1,2 GB no total (NT 300 MB + AT 900 MB), 1189 capítulos.
O download vem da internet do Colab, não da sua — seu PC não entra nisso.

**É seguro rodar de novo:** capítulo que já está no Drive é pulado. Se a
sessão cair no meio, roda de novo que ele continua de onde parou.

---
*A gravação foi liberada em domínio público por David Williams; o texto WEB é
de Michael Paul Johnson, também sem restrição de copyright.*

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — Drive e módulos (rode uma vez por sessão)             ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Monta o Drive (desmonta antes, pra não travar em sessão presa) ──────────
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive montado')

# ── Copia os módulos do Drive pra /content/pipeline ─────────────────────────
import shutil, sys
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"   # fixo pro projeto todo
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO_MODULOS = Path("/content/pipeline")

if not PASTA_MODULOS.exists():
    raise SystemExit(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")

if DESTINO_MODULOS.exists():
    shutil.rmtree(DESTINO_MODULOS)
shutil.copytree(PASTA_MODULOS, DESTINO_MODULOS)
print(f"✅ {len(list(DESTINO_MODULOS.glob('*.py')))} módulos copiados")

if str(DESTINO_MODULOS) not in sys.path:
    sys.path.insert(0, str(DESTINO_MODULOS))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO — edite só esta célula                          ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. O QUE BAIXAR ─────────────────────────────────────────────────────────
# "ambos"  -> Bíblia inteira (1189 capítulos, ~1,2 GB)  ← recomendado, roda 1x
# "nt"     -> só Novo Testamento (260 capítulos, ~300 MB)
# "at"     -> só Antigo Testamento (929 capítulos, ~900 MB)
BAIXAR = "ambos"

# ── 2. ONDE SALVAR NO DRIVE ─────────────────────────────────────────────────
# Fica em assets/ porque é material COMPARTILHADO entre todos os vídeos --
# mesma lógica de assets/trilha e assets/marca.
SUBPASTA_DESTINO = "biblia_audio"     # -> narrated_video/assets/biblia_audio/

# ── 3. REFAZER O QUE JÁ ESTÁ LÁ? ────────────────────────────────────────────
# False = pula capítulo que já existe no Drive (o normal, e o que faz o
#         notebook ser seguro de rodar de novo depois de uma queda).
# True  = reescreve tudo. Só use se desconfiar de arquivo corrompido.
REFAZER = False

# ── 4. FONTE (não mexa sem motivo) ──────────────────────────────────────────
BASE_URL = "https://audiotreasure.com/content/WEBD_AT/zipfiles"
ZIPS = {
    "nt": ("WEB_NT_Audio.zip", 250_000_000),   # (arquivo, tamanho mínimo esperado)
    "at": ("WEB_OT_Audio.zip", 800_000_000),
}

print(f"Baixar ......... {BAIXAR}")
print(f"Destino ........ narrated_video/assets/{SUBPASTA_DESTINO}/")
print(f"Refazer ........ {REFAZER}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INICIALIZAR                                                   ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

import biblia_livros as bl

PASTA_DESTINO = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/assets/{SUBPASTA_DESTINO}")
PASTA_DESTINO.mkdir(parents=True, exist_ok=True)

PASTA_ZIPS   = Path("/content/biblia_zips");   PASTA_ZIPS.mkdir(exist_ok=True)
PASTA_BRUTOS = Path("/content/biblia_brutos"); PASTA_BRUTOS.mkdir(exist_ok=True)

if BAIXAR == "ambos":
    _chaves = ["nt", "at"]
    LIVROS_ALVO = bl.LIVROS
elif BAIXAR == "nt":
    _chaves = ["nt"]
    LIVROS_ALVO = bl.novo_testamento()
elif BAIXAR == "at":
    _chaves = ["at"]
    LIVROS_ALVO = bl.antigo_testamento()
else:
    raise ValueError(f'BAIXAR deve ser "ambos", "nt" ou "at" -- veio {BAIXAR!r}')

CAPITULOS_ALVO = list(bl.todos_capitulos(LIVROS_ALVO))

print(f"📖 {len(LIVROS_ALVO)} livros, {len(CAPITULOS_ALVO)} capítulos")
print(f"📁 {PASTA_DESTINO}")
_ja = sum(1 for l, c in CAPITULOS_ALVO if (PASTA_DESTINO / f"{l.nome_projeto(c)}.mp3").exists())
print(f"✅ já no Drive: {_ja}  |  faltando: {len(CAPITULOS_ALVO) - _ja}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⬇️  1/3 — BAIXAR OS ZIPS                                         ║
# ║  wget -c: se cair no meio, roda de novo que ele CONTINUA          ║
# ╚══════════════════════════════════════════════════════════════════╝

for chave in _chaves:
    nome_zip, tamanho_min = ZIPS[chave]
    destino = PASTA_ZIPS / nome_zip

    if destino.exists() and destino.stat().st_size >= tamanho_min:
        print(f"⏭️  {nome_zip} já baixado ({destino.stat().st_size/1e6:.0f} MB)")
        continue

    print(f"⬇️  {nome_zip} ...")
    !wget -c -q --show-progress -O "{destino}" "{BASE_URL}/{nome_zip}"

    # Guarda: HTML de erro/redirect entra como "arquivo baixado" e só quebraria
    # lá na frente, na descompactação, com uma mensagem que não ajuda em nada.
    tamanho = destino.stat().st_size if destino.exists() else 0
    if tamanho < tamanho_min:
        raise SystemExit(
            f"❌ {nome_zip} veio com {tamanho/1e6:.1f} MB, esperava pelo menos "
            f"{tamanho_min/1e6:.0f} MB.\\n"
            f"   Provavelmente o link mudou ou o servidor respondeu erro.\\n"
            f"   Confira em {BASE_URL}/")
    print(f"✅ {nome_zip} — {tamanho/1e6:.0f} MB")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 2/3 — DESCOMPACTAR E INDEXAR                                  ║
# ╚══════════════════════════════════════════════════════════════════╝

import zipfile

for chave in _chaves:
    nome_zip, _ = ZIPS[chave]
    print(f"📦 abrindo {nome_zip} ...")
    with zipfile.ZipFile(PASTA_ZIPS / nome_zip) as z:
        z.extractall(PASTA_BRUTOS)

# A estrutura interna do zip não é garantida (pode ter subpasta por livro, ou
# não), então varre a árvore inteira.
#
# E indexa por (livro, capítulo), NÃO pelo nome do arquivo. O nome a fonte não
# garante: `Prov` ou `Proverbs`, `Lam1` ou `Lamentations01`, `Soloman` ou
# `Solomon`. Na primeira execução deste notebook, 120 capítulos saíram como
# "faltando" e estavam todos dentro do zip, só com outro nome. O que a fonte
# garante é a numeração canônica -- e ela está no começo e no fim de todo
# arquivo. Ver biblia_livros.chave_audio.
caminho_por_stem = {c.stem.strip().lower(): c for c in PASTA_BRUTOS.rglob("*.mp3")}
indice, ignorados, colisoes = bl.indexar_por_chave(caminho_por_stem)

print(f"🎧 {len(caminho_por_stem)} mp3 encontrados, {len(indice)} indexados por (livro, capítulo)")

if ignorados:
    print(f"\n⚠️  {len(ignorados)} arquivo(s) sem numeração reconhecível — ignorados:")
    for stem in ignorados[:20]:
        print(f"   {stem}.mp3")
    if len(ignorados) > 20:
        print(f"   ... e mais {len(ignorados) - 20}")

if colisoes:
    # Dois arquivos disputando o mesmo capítulo quer dizer que a fonte mudou de
    # forma. O primeiro fica, mas você TEM que ver quais foram -- escolher em
    # silêncio aqui daria um vídeo lendo outro capítulo, sem erro nenhum.
    print(f"\n🚨 {len(colisoes)} capítulo(s) com MAIS DE UM arquivo candidato:")
    for (num, cap), stems in sorted(colisoes.items()):
        print(f"   {bl.por_numero(num).nome_projeto(cap)}: {', '.join(stems)}")
    print("   → o primeiro de cada lista foi usado. Confira antes de confiar.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📤 3/3 — RENOMEAR E COPIAR PRO DRIVE                             ║
# ║  Copiar milhares de arquivos pequenos pro Drive é lento --        ║
# ║  ~10 a 20 min pra Bíblia inteira. Deixe rodando.                  ║
# ╚══════════════════════════════════════════════════════════════════╝

import shutil

copiados, pulados = 0, 0
faltando = []          # capítulo do cânone que o zip não tinha
usados = set()         # chaves que casaram, pra achar o que sobrou

for i, (livro, cap) in enumerate(CAPITULOS_ALVO, 1):
    chave_cap = livro.chave(cap)
    destino = PASTA_DESTINO / f"{livro.nome_projeto(cap)}.mp3"

    if destino.exists() and not REFAZER:
        pulados += 1
        usados.add(chave_cap)
        continue

    stem_origem = indice.get(chave_cap)
    if stem_origem is None:
        faltando.append((livro, cap))
        continue

    shutil.copy2(caminho_por_stem[stem_origem], destino)
    usados.add(chave_cap)
    copiados += 1

    if i % 100 == 0:
        print(f"   ... {i}/{len(CAPITULOS_ALVO)}")

print()
print(f"✅ copiados ... {copiados}")
print(f"⏭️  já estavam . {pulados}")
print(f"❌ faltando ... {len(faltando)}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 CONFERÊNCIA — o que faltou e o que sobrou                     ║
# ╚══════════════════════════════════════════════════════════════════╝

# Duas listas:
#   FALTANDO -> capítulo que o cânone tem e o zip não entregou
#   SOBRANDO -> mp3 do zip que nenhum capítulo reclamou
#
# Antes, um livro aparecia nas DUAS quando o nome na fonte era diferente do
# que a tabela supunha -- foi o que aconteceu com 120 capítulos. Agora o
# casamento é por (livro, capítulo), então isso não acontece mais: FALTANDO
# quer dizer ausente de verdade, e SOBRANDO quer dizer material fora do cânone
# (introduções, faixas extras).

if faltando:
    print(f"❌ FALTANDO ({len(faltando)}) — não estão no zip:")
    for livro, cap in faltando:
        print(f"   {livro.nome_projeto(cap):<16} (esperava algo como '{livro.stem_audio(cap).lower()}.mp3')")
else:
    print("✅ nenhum capítulo faltando")

print()
sobrando = sorted(stem for chave, stem in indice.items() if chave not in usados)
if sobrando:
    print(f"⚠️  SOBRANDO ({len(sobrando)}) — mp3 de capítulo que este download não pediu:")
    for stem in sobrando[:40]:
        print(f"   {stem}.mp3")
    if len(sobrando) > 40:
        print(f"   ... e mais {len(sobrando) - 40}")
    if BAIXAR != "ambos":
        print(f'   (normal: BAIXAR="{BAIXAR}", então o outro testamento sobra)')
else:
    print("✅ nada sobrando")

print()
total_drive = len(list(PASTA_DESTINO.glob("*.mp3")))
tamanho_gb = sum(p.stat().st_size for p in PASTA_DESTINO.glob("*.mp3")) / 1e9
print(f"📁 {PASTA_DESTINO}")
print(f"   {total_drive} capítulos, {tamanho_gb:.2f} GB")
if BAIXAR == "ambos":
    print(f"   cânone completo = {bl.TOTAL_CAPITULOS} capítulos")
    if total_drive == bl.TOTAL_CAPITULOS:
        print("   ✅ completo")
    else:
        print(f"   ⚠️  faltam {bl.TOTAL_CAPITULOS - total_drive}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🧹 OPCIONAL — liberar o disco do Colab                           ║
# ║  Só os arquivos LOCAIS. O Drive não é tocado.                     ║
# ╚══════════════════════════════════════════════════════════════════╝

import shutil
for pasta in (PASTA_ZIPS, PASTA_BRUTOS):
    if pasta.exists():
        shutil.rmtree(pasta)
        print(f"🧹 {pasta}")
print("✅ disco local liberado (o que está no Drive continua lá)")